# RAG with PDF using LangChain

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) system using LangChain to answer questions based on a PDF document.

In [1]:
import os
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_openai import ChatOpenAI 

/home/miad/projects/RAG/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setup Environment
Define API keys for OpenRouter.

In [2]:
# SETUP: OpenRouter
# Get key from: https://openrouter.ai/keys
os.environ["OPENAI_API_KEY"] = "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" # Your OpenRouter Key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

## Prerequisites: LLM and Embedding Model

In [3]:
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
llm = ChatOpenAI(
    #model="meta-llama/llama-3-8b-instruct",
    model="x-ai/grok-4.1-fast",
    temperature=0.1
)

## 1. Load PDF

Load the PDF document using `PyPDFLoader`.

In [4]:
# 1. Load PDF
loader = PyPDFLoader("./history_of_video_games.pdf")
docs = loader.load()

In [5]:
len(docs)

8

## 2. Split & Embed

Split the loaded document into smaller chunks and create a vector store using Chroma and the embedding model.

In [10]:
# 2. Split & Embed
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)

## 3. Retriever with MMR (Maximal Marginal Relevance)

Create a retriever that uses MMR to select diverse chunks, avoiding redundancy in the retrieved context.

In [11]:
# 3. Retriever with MMR (Maximal Marginal Relevance)
# This helps avoid getting 4 chunks that say the exact same thing
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4})

## 4. Create Chain with Sources

Define the system prompt and create the question-answering chain. This chain combines the retrieved documents and the LLM to generate an answer.

In [12]:
# 4. Create Chain with Sources
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

## 5. Run

Invoke the chain with a query and print the answer along with the source documents.

In [13]:
# 5. Run
response = rag_chain.invoke({"input": "Who are the specific team members responsible for creating the iOS game Star Trux, and what specific roles did each person perform during its development?"})

print("Answer:", response["answer"])
print("\n--- SOURCES ---")
for i, doc in enumerate(response["context"]):
    # Metadata usually contains page numbers for PDFs
    print(f"Source {i+1} (Page {doc.metadata.get('page', 'Unknown')}):")
    print(doc.page_content[:100] + "...")

Answer: **John Powers** handled engineering, **Ken Balthaser** managed design, and **Rob Powers** was responsible for art in developing the iOS game Star Trux. These three family friends, with nearly 100 years of combined experience, created the game over one year. No other team members are mentioned.

--- SOURCES ---
Source 1 (Page 4):
Rob Powers (Sega, WestWood), Ken Balthaser Jr.(Sega and EA) and Neil
Balthaser(Alexandria, Capcom). ...
Source 2 (Page 4):
and their friends swapping stories about late-night programming sessions and
games that got done and...
Source 3 (Page 0):
Busine ss Comme nt ary
Featured Blog | This community-written post highlights the best of what the g...
Source 4 (Page 1):
In the 1970's a small group of people formed a company called Authorship Resource
Inc.  This company...
